In [ ]:
import pandas as pd
import os
from pathlib import Path
from src.common.minio_client import download_df_parquet

In [ ]:
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

## Dataset final

In [ ]:
df_final = download_df_parquet(access_key, secret_key,"grupo5/final/year=2025/month=01/dataset_final.parquet")


In [ ]:
df_final.stop_id.value_counts()

In [ ]:
(df_final[df_final.is_unscheduled == False].stop_id.value_counts() > 0).sum()

In [ ]:
df_final['stop_id'].str[:3].value_counts()

In [ ]:
df_final.columns

## GTFS Scheduled

In [ ]:
START_DATE = "2025-01-06"
END_DATE   = "2025-01-19"  # 2 semanas bastan para capturar toda la topología del grafo

dates = pd.date_range(start=START_DATE, end=END_DATE).strftime("%Y-%m-%d").tolist()
dfs = []
for date in dates:
    try:
        df_gtfs = download_df_parquet(
            access_key, secret_key,
            f"grupo5/cleaned/gtfs_clean_scheduled/date={date}/gtfs_scheduled_{date}.parquet"
        )
        dfs.append(df_gtfs)
    except Exception:
        print(f"No disponible: {date}")
        continue

if not dfs:
    raise RuntimeError("No se pudo descargar ningún archivo GTFS. Verifica las credenciales y la ruta.")

df = pd.concat(dfs, ignore_index=True)
del dfs  # liberar lista de DataFrames intermedios
print(f"GTFS cargado: {len(df):,} filas, {df['trip_uid'].nunique():,} viajes únicos")

## Cálculo de Matrices de adyacencia

In [ ]:
import numpy as np
import gc

# 1. Ordenar cronológicamente por viaje
df = df.sort_values(by=['trip_uid', 'scheduled_seconds']).reset_index(drop=True)

# 2. Calcular la parada siguiente dentro de cada viaje
df['next_stop_id']           = df.groupby('trip_uid')['stop_id'].shift(-1)
df['next_scheduled_seconds'] = df.groupby('trip_uid')['scheduled_seconds'].shift(-1)

# 3. Construir edges_df y liberar df inmediatamente para recuperar memoria
edges_df = df.dropna(subset=['next_stop_id']).copy()
del df
gc.collect()

# 4. Tiempo de viaje entre paradas consecutivas
edges_df['travel_time'] = edges_df['next_scheduled_seconds'] - edges_df['scheduled_seconds']

In [ ]:
# Eliminar tiempos de viaje inválidos
edges_df = edges_df[edges_df['travel_time'] > 0]

# Agregar aristas únicas con su tiempo mediano de viaje
graph_df = edges_df.groupby(['stop_id', 'next_stop_id']).agg(
    median_travel_time=('travel_time', 'median'),
    trip_count=('trip_uid', 'count')
).reset_index()

# edges_df ya no se necesita
del edges_df
gc.collect()

print(f"Aristas en el grafo: {len(graph_df):,}")

In [ ]:
# Nodos únicos: unión de orígenes y destinos de todas las aristas
nodes = sorted(set(graph_df['stop_id'].unique()) | set(graph_df['next_stop_id'].unique()))
n_nodes = len(nodes)

node_to_idx = {stop_id: idx for idx, stop_id in enumerate(nodes)}

# Vectorised fill — evita el loop lento de iterrows
src_idx = graph_df['stop_id'].map(node_to_idx).values
dst_idx = graph_df['next_stop_id'].map(node_to_idx).values

A_unweighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)
A_unweighted[src_idx, dst_idx] = 1.0

print("Unweighted Adjacency Matrix Shape:", A_unweighted.shape)

In [ ]:
# Matriz de adyacencia ponderada (Gaussian Kernel sobre tiempo de viaje)
distances = graph_df['median_travel_time'].values
sigma = distances.std()

# Vectorised fill con guarda epsilon en denominador (evita sigma≈0 si todos los tiempos son iguales)
weights = np.exp(-(distances ** 2) / (sigma ** 2 + 1e-6))

A_weighted = np.zeros((n_nodes, n_nodes), dtype=np.float32)
A_weighted[src_idx, dst_idx] = weights

del graph_df
gc.collect()

print("Weighted Adjacency Matrix Shape:", A_weighted.shape)

In [ ]:
A_unweighted.sum()

In [ ]:
# scheduled_nodes_list ya está disponible como `nodes` del paso anterior
scheduled_nodes_list = nodes
print(f"Nodos en el grafo: {len(scheduled_nodes_list)}")

## Tensor generation

In [ ]:
import numpy as np

def build_spatiotemporal_tensor(df: pd.DataFrame, scheduled_nodes: list[str], freq: str = '15min'):
    """
    Transforms a flat event DataFrame into a (Time, Nodes, Features) tensor.
    Features: delay_seconds, lagged_delay_1, lagged_delay_2, is_unscheduled,
              temp_extreme, n_eventos_afectando, route_rolling_delay,
              actual_headway_seconds, hour_sin, hour_cos, dow  (11 total)
    Targets:  target_delay_10m, target_delay_20m, target_delay_30m
    """
    # 1. Filtrar primero (subconjunto pequeño) y solo entonces copiar
    df = df[df['stop_id'].isin(scheduled_nodes)].copy()
    df['time_bin'] = pd.to_datetime(df['merge_time']).dt.floor(freq)

    # 2. Agregar eventos por ventana temporal y estación
    agg_rules = {
        'delay_seconds':          'mean',
        'lagged_delay_1':         'mean',
        'lagged_delay_2':         'mean',
        'is_unscheduled':         'sum',
        'temp_extreme':           'max',
        'n_eventos_afectando':    'max',
        'route_rolling_delay':    'mean',
        'actual_headway_seconds': 'mean',
        'target_delay_10m':       'mean',
        'target_delay_20m':       'mean',
        'target_delay_30m':       'mean',
    }
    grouped_df = df.groupby(['time_bin', 'stop_id']).agg(agg_rules)
    del df
    gc.collect()

    # 3. Grid completo (producto cartesiano tiempo × nodos)
    all_times = pd.date_range(
        start=grouped_df.index.get_level_values('time_bin').min(),
        end=grouped_df.index.get_level_values('time_bin').max(),
        freq=freq
    )
    all_nodes = sorted(scheduled_nodes)
    full_index = pd.MultiIndex.from_product([all_times, all_nodes], names=['time_bin', 'stop_id'])
    full_df = grouped_df.reindex(full_index).reset_index()
    del grouped_df
    gc.collect()

    # 4. Imputación
    zero_fill_cols = ['delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
                      'is_unscheduled', 'route_rolling_delay', 'actual_headway_seconds']
    full_df[zero_fill_cols] = full_df[zero_fill_cols].fillna(0)

    context_cols = ['temp_extreme', 'n_eventos_afectando']
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].ffill()
    full_df[context_cols] = full_df.groupby('stop_id')[context_cols].bfill()

    # 5. Features temporales derivadas del time_bin (sin ruido de agregación)
    full_df['hour_sin'] = np.sin(2 * np.pi * full_df['time_bin'].dt.hour / 24)
    full_df['hour_cos'] = np.cos(2 * np.pi * full_df['time_bin'].dt.hour / 24)
    full_df['dow']      = full_df['time_bin'].dt.dayofweek.astype(float)

    full_df = full_df.sort_values(['time_bin', 'stop_id'])

    feature_cols = [
        'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
        'is_unscheduled', 'temp_extreme', 'n_eventos_afectando',
        'route_rolling_delay', 'actual_headway_seconds',
        'hour_sin', 'hour_cos', 'dow'
    ]
    target_cols = ['target_delay_10m', 'target_delay_20m', 'target_delay_30m']

    T = len(all_times)
    N = len(all_nodes)

    # 6. Extraer arrays y liberar full_df antes de devolver
    X_tensor = full_df[feature_cols].values.reshape(T, N, len(feature_cols))
    Y_tensor = full_df[target_cols].values.reshape(T, N, len(target_cols))
    del full_df
    gc.collect()

    return X_tensor, Y_tensor, all_times, all_nodes

In [ ]:
import gc

X, Y, times, nodes = build_spatiotemporal_tensor(df_final, scheduled_nodes_list)
print("X Shape (Time, Nodes, Features):", X.shape)
print("Y Shape (Time, Nodes, Targets):", Y.shape)

# df_final ya no se necesita
del df_final
gc.collect()

## DCRNN Training

In [ ]:
# División cronológica (Train / Val / Test)
import numpy as np
import gc

X_np = np.nan_to_num(np.asarray(X), nan=0.0)
Y_np = np.nan_to_num(np.asarray(Y), nan=0.0)
del X, Y
gc.collect()

T = X_np.shape[0]
train_end = int(T * 0.70)
val_end   = train_end + int(T * 0.10)

X_train, X_val, X_test = X_np[:train_end], X_np[train_end:val_end], X_np[val_end:]
Y_train, Y_val, Y_test = Y_np[:train_end], Y_np[train_end:val_end], Y_np[val_end:]
del X_np, Y_np
gc.collect()

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Y_train: {Y_train.shape} | Y_val: {Y_val.shape} | Y_test: {Y_test.shape}")

In [ ]:
# Normalización de características y target (Feature & Target Scaling)
from sklearn.preprocessing import StandardScaler
import gc

# --- Escalar X ---
T_train, N, F = X_train.shape
X_train_2d = X_train.reshape(-1, F)
X_val_2d   = X_val.reshape(-1, F)
X_test_2d  = X_test.reshape(-1, F)

scaler_X = StandardScaler()
scaler_X.fit(X_train_2d)

X_train_scaled = scaler_X.transform(X_train_2d).reshape(X_train.shape)
X_val_scaled   = scaler_X.transform(X_val_2d).reshape(X_val.shape)
X_test_scaled  = scaler_X.transform(X_test_2d).reshape(X_test.shape)
del X_train, X_val, X_test, X_train_2d, X_val_2d, X_test_2d

# --- Escalar Y por separado ---
H_out = Y_train.shape[-1]
Y_train_2d = Y_train.reshape(-1, H_out)
Y_val_2d   = Y_val.reshape(-1, H_out)
Y_test_2d  = Y_test.reshape(-1, H_out)

scaler_Y = StandardScaler()
scaler_Y.fit(Y_train_2d)

Y_train_scaled = scaler_Y.transform(Y_train_2d).reshape(Y_train.shape)
Y_val_scaled   = scaler_Y.transform(Y_val_2d).reshape(Y_val.shape)
Y_test_scaled  = scaler_Y.transform(Y_test_2d).reshape(Y_test.shape)
del Y_train, Y_val, Y_test, Y_train_2d, Y_val_2d, Y_test_2d
gc.collect()

print("Escalado completado")
print(f"X_train_scaled: {X_train_scaled.shape}  (features: {F})")
print(f"Y_train_scaled: {Y_train_scaled.shape}  (targets: {H_out})")

In [ ]:
# Creacion de ventana deslizante (PyTorch Dataset)
import torch
from torch.utils.data import Dataset

class SubwayDataset(Dataset):
    def __init__(self, X, Y, history_len=4):
        self.X = torch.as_tensor(X.copy(), dtype=torch.float32)
        self.Y = torch.as_tensor(Y.copy(), dtype=torch.float32)
        self.history_len = history_len

        if self.X.shape[0] != self.Y.shape[0]:
            raise ValueError("X e Y deben tener el mismo tamano temporal (eje T).")
        if self.history_len < 1:
            raise ValueError("history_len debe ser >= 1.")
        if self.X.shape[0] <= self.history_len:
            raise ValueError("T debe ser mayor que history_len para generar ventanas.")

    def __len__(self):
        # Cada muestra usa [t, ..., t+history_len-1] para predecir t+history_len
        return self.X.shape[0] - self.history_len

    def __getitem__(self, idx):
        x_window = self.X[idx : idx + self.history_len]          # (history_len, N, F)
        y_window = self.Y[idx + self.history_len].unsqueeze(0)   # (1, N, H)
        return x_window, y_window

In [ ]:
# Generación de DataLoaders
from torch.utils.data import DataLoader

history_len = 8   # 2 horas de contexto a bins de 15min
batch_size  = 32

train_dataset = SubwayDataset(X_train_scaled, Y_train_scaled, history_len=history_len)
val_dataset   = SubwayDataset(X_val_scaled,   Y_val_scaled,   history_len=history_len)
test_dataset  = SubwayDataset(X_test_scaled,  Y_test_scaled,  history_len=history_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}  # (Batch, History, Nodes, Features)")
print(f"y_batch shape: {y_batch.shape}  # (Batch, 1, Nodes, Horizontes)")

In [ ]:
# Preparación del Grafo (PyTorch Geometric Format)
import torch

print('Cuda available:', torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
A_tensor = torch.as_tensor(A_weighted, dtype=torch.float32)

# Matriz densa → representación COO: edge_index [2, E], edge_weight [E]
edge_index = torch.nonzero(A_tensor != 0, as_tuple=False).t().contiguous()
edge_weight = A_tensor[edge_index[0], edge_index[1]].contiguous()
del A_tensor

from torch_geometric.utils import add_self_loops

# Auto-conexiones con peso 1.0 para evitar nans en la convolución
edge_index, edge_weight = add_self_loops(
    edge_index,
    edge_weight,
    fill_value=1.0,
    num_nodes=n_nodes
)

edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)

print(f"Device: {device}")
print(f"edge_index shape: {edge_index.shape}")
print(f"edge_weight shape: {edge_weight.shape}")
print(f"Número de aristas (con self-loops): {edge_weight.numel()}")

In [ ]:
# Definición de la Arquitectura del Modelo
import torch.nn as nn
from torch_geometric_temporal.nn.recurrent import DCRNN

class SubwayDCRNN(nn.Module):
    def __init__(self, in_channels=11, hidden_channels=64, out_horizons=3, K=2, dropout=0.1):
        super().__init__()
        self.dcrnn = DCRNN(in_channels=in_channels, out_channels=hidden_channels, K=K)
        self.readout = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_horizons)
        )

    def forward(self, X, edge_index, edge_weight):
        # X: (Batch, History, Nodes, Features)
        B, H, N, F = X.shape
        hidden_state = None

        # DCRNN de torch_geometric_temporal opera sobre un solo grafo (N, F),
        # por lo que iteramos sobre el batch — correcto en semántica, costoso en GPU
        for t in range(H):
            x_t = X[:, t, :, :]  # (Batch, Nodes, Features)
            next_hidden = []
            for b in range(B):
                h_prev_b = None if hidden_state is None else hidden_state[b]
                h_b = self.dcrnn(x_t[b], edge_index, edge_weight, h_prev_b)
                next_hidden.append(h_b)
            hidden_state = torch.stack(next_hidden, dim=0)  # (Batch, Nodes, hidden)

        y_hat = self.readout(hidden_state)  # (Batch, Nodes, out_horizons)
        y_hat = y_hat.unsqueeze(1)          # (Batch, 1, Nodes, out_horizons)
        return y_hat

In [ ]:
# Instanciación y Configuración
model = SubwayDCRNN(in_channels=11, hidden_channels=64, out_horizons=3, K=2, dropout=0.1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, min_lr=1e-5
)
criterion = torch.nn.L1Loss()  # MAE en espacio escalado

print(model)
print(f"\nDevice: {device}")
print(f"Optimizador: Adam | lr=0.001")
print(f"Scheduler: ReduceLROnPlateau (patience=3, factor=0.5)")
print(f"Loss: L1Loss (MAE) | Targets: target_delay_10m / 20m / 30m")

In [ ]:
print("=== Shape and Sanity Checks ===")
print(f"\nX tensors:")
print(f"  X_train_scaled: {X_train_scaled.shape}")
print(f"  X_val_scaled:   {X_val_scaled.shape}")
print(f"  X_test_scaled:  {X_test_scaled.shape}")

print(f"\nY tensors (scaled):")
print(f"  Y_train_scaled: {Y_train_scaled.shape}")
print(f"  Y_val_scaled:   {Y_val_scaled.shape}")
print(f"  Y_test_scaled:  {Y_test_scaled.shape}")

print(f"\nDataset windows (con history_len={history_len}):")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples:   {len(val_dataset)}")
print(f"  Test samples:  {len(test_dataset)}")

print(f"\nNaN checks:")
print(f"  X_train has NaNs: {np.isnan(X_train_scaled).any()}")
print(f"  Y_train has NaNs: {np.isnan(Y_train_scaled).any()}")
print(f"  X_test has NaNs:  {np.isnan(X_test_scaled).any()}")
print(f"  Y_test has NaNs:  {np.isnan(Y_test_scaled).any()}")

x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch dimensions:")
print(f"  x_batch: {tuple(x_sample.shape)}  # (Batch, History, Nodes, Features)")
print(f"  y_batch: {tuple(y_sample.shape)}  # (Batch, 1, Nodes, Horizontes)")

In [ ]:
# Bucle de Entrenamiento con Early Stopping
import copy

num_epochs    = 50
es_patience   = 7       # épocas sin mejora antes de parar
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state  = None

for epoch in range(1, num_epochs + 1):
    model.train()
    train_running_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(x_batch, edge_index, edge_weight)
        loss   = criterion(y_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        optimizer.step()
        train_running_loss += loss.item()

    train_loss = train_running_loss / max(len(train_loader), 1)

    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            val_running_loss += criterion(model(x_batch, edge_index, edge_weight), y_batch).item()

    val_loss = val_running_loss / max(len(val_loader), 1)

    prev_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    curr_lr = optimizer.param_groups[0]['lr']
    lr_tag  = f"  ↓ lr={curr_lr:.2e}" if curr_lr < prev_lr else ""

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:02d}/{num_epochs} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{lr_tag}"
          + (f"  [best]" if epochs_no_improve == 0 else ""))

    if epochs_no_improve >= es_patience:
        print(f"\nEarly stopping en época {epoch} (sin mejora en {es_patience} épocas).")
        break

# Restaurar el mejor modelo
model.load_state_dict(best_model_state)
print(f"\nMejor val loss: {best_val_loss:.6f}")

In [ ]:
# Evaluación en test: MAE en segundos reales (invirtiendo el escalado de Y)
import numpy as np

model.eval()
preds_list, trues_list = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        y_pred  = model(x_batch, edge_index, edge_weight)  # (B, 1, N, 3) escalado
        preds_list.append(y_pred.cpu().numpy())
        trues_list.append(y_batch.numpy())

preds = np.concatenate(preds_list, axis=0).squeeze(1)  # (T_test, N, 3)
trues = np.concatenate(trues_list, axis=0).squeeze(1)  # (T_test, N, 3)

# Invertir escalado
T_test, N_test, H_out = preds.shape
preds_inv = scaler_Y.inverse_transform(preds.reshape(-1, H_out)).reshape(T_test, N_test, H_out)
trues_inv = scaler_Y.inverse_transform(trues.reshape(-1, H_out)).reshape(T_test, N_test, H_out)

horizons = ['10m', '20m', '30m']
print("MAE en segundos reales (test set):")
for i, h in enumerate(horizons):
    mae = np.abs(preds_inv[:, :, i] - trues_inv[:, :, i]).mean()
    print(f"  target_delay_{h}: {mae:.1f} s")